In [ ]:
import pandas as pd
import warnings
import joblib
import seaborn as sns
import matplotlib.pyplot as plt
%run strategy_creation.ipynb
%run features_implementation.ipynb
%run model_research.ipynb
%run baynesian_opt.ipynb
%run splitted_df_validation.ipynb

In [ ]:
df = pd.read_csv('btc_3yrs_wfeatures.csv')
df_trades = pd.read_csv('df_trades.csv')
model = joblib.load('dump_model.pk1')

steps i have to take and logic
the model is already train more or less on the 3/4 of half df_trades, let's approx on about 40% of the whole df_trades, so first step is to discard it

In [ ]:
df_trades_len = len(df_trades)
print(f'df_trades len is: {df_trades_len}')
print(f'new df should start from line: {int(round(df_trades_len*0.4))}')

cut_idx = int(round(df_trades_len*0.4))
new_df_trades = df_trades[cut_idx:]
print(f'new df len is : {len(new_df_trades)}')

nex step will be comparing results from og df_test with splitted section of new_df_trades to check how model reacts and if and how the results degrade over time, for statistical purpose each split will be not less then 500 trades

In [ ]:
cut_len = 100
start_idx = 0
end_idx = 100
cut_numbers = int(round(len(new_df_trades) / cut_len))
print(f'the df will be split in {cut_numbers} parts')
results_list = []


In [ ]:
#also get model features
features = model.get_booster().feature_names

In [ ]:
for cuts in range(cut_numbers):
    #define the split
    end_idx = start_idx + cut_len
    current_split = new_df_trades[start_idx:end_idx]
    #test model for each split
    base_precision, base_recall = test_model(df = current_split , model = model, features = features)
    
    results_list.append({
        'cut_number': cuts,
        'start_idx': start_idx,
        'end_Index': end_idx,
        'num_nows': len(current_split),
        'precision': base_precision,
        'recall': base_recall
    })
    
    start_idx = end_idx
    
    #print(f" Cut {cuts+1}: Precision={base_precision:.4f}, Recall={base_recall:.4f}")

    results_df = pd.DataFrame(results_list)

In [ ]:
results_df

In [ ]:
results_df['precision'].mean()

In [ ]:
sns.set_style("whitegrid")
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1) # Sets up a 1 row, 2 column grid for plots, targeting the 1st position
sns.lineplot(
    x='cut_number', 
    y='precision', 
    data=results_df, 
    marker='o', 
    color='tab:blue', 
    label='Precision Score'
)

plt.subplot(1, 2, 1) # Sets up a 1 row, 2 column grid for plots, targeting the 1st position
sns.lineplot(
    x='cut_number', 
    y='precision', 
    data=results_df, 
    marker='o', 
    color='tab:blue', 
    label='Precision Score'
)

plt.subplot(1, 2, 2) # Targets the 2nd position in the grid
sns.lineplot(
    x='cut_number', 
    y='recall', 
    data=results_df, 
    marker='o', 
    color='tab:green',
    label='Recall Score'
)

# Add horizontal line for average recall
avg_recall = results_df['recall'].mean()
plt.axhline(avg_recall, color='red', linestyle='--', label=f'Avg Recall ({avg_recall:.3f})')

plt.title('Model Recall Across Sequential Data Cuts', fontsize=14)
plt.xlabel('Data Cut Number (Time/Sequence)', fontsize=12)
plt.ylabel('Recall Score', fontsize=12)
plt.ylim(0.35, 0.6) # Maintain consistent Y-limit
plt.legend()

# Adjust layout to prevent overlap and show the plots
plt.tight_layout()
plt.show()
#

now we'll get model info to retrain the model keeping old params

In [ ]:
model = joblib.load('dump_model.pk1')
old_model = model
old_params = model.get_params()
old_booster = old_model.get_booster()
features = old_booster.feature_names

the split logic will have to be different, lets first test how the model react only being retraind on the single split data

In [ ]:
start_idx = 0
end_idx = 500
results_list = []

for cuts in range(cut_numbers):
    #define the split
    end_idx = start_idx + cut_len
    current_split = new_df_trades[start_idx:end_idx]

    if cuts < 1:
        #test model for each split
        base_precision, base_recall = test_model(df = current_split , model = model, features = features)
        
        results_list.append({
            'cut_number': cuts,
            'start_idx': start_idx,
            'end_Index': end_idx,
            'num_nows': len(current_split),
            'precision': base_precision,
            'recall': base_recall
        })

    
    if cuts > 0:
        train_split = new_df_trades[(start_idx - 500) : (end_idx - 500)]
        X_train = train_split[features]
        y_train = train_split['Trades']

        old_model = model 
        old_params = old_model.get_params()
        old_booster = old_model.get_booster()
        model = XGBClassifier(**old_params)
        model.fit(X_train, y_train, xgb_model = old_booster)
        

        base_precision, base_recall = test_model(df = current_split , model = model, features = features)
        
        results_list.append({
            'cut_number': cuts,
            'start_idx': start_idx,
            'end_Index': end_idx,
            'num_nows': len(current_split),
            'precision': base_precision,
            'recall': base_recall
        })

        print(f" Cut {cuts+1}: Precision={base_precision:.4f}, Recall={base_recall:.4f}")






    start_idx = end_idx
    
    results_df = pd.DataFrame(results_list)

In [ ]:
sns.set_style("whitegrid")
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1) # Sets up a 1 row, 2 column grid for plots, targeting the 1st position
sns.lineplot(
    x='cut_number', 
    y='precision', 
    data=results_df, 
    marker='o', 
    color='tab:blue', 
    label='Precision Score'
)

plt.subplot(1, 2, 1) # Sets up a 1 row, 2 column grid for plots, targeting the 1st position
sns.lineplot(
    x='cut_number', 
    y='precision', 
    data=results_df, 
    marker='o', 
    color='tab:blue', 
    label='Precision Score'
)

plt.subplot(1, 2, 2) # Targets the 2nd position in the grid
sns.lineplot(
    x='cut_number', 
    y='recall', 
    data=results_df, 
    marker='o', 
    color='tab:green',
    label='Recall Score'
)

# Add horizontal line for average recall
avg_recall = results_df['recall'].mean()
plt.axhline(avg_recall, color='red', linestyle='--', label=f'Avg Recall ({avg_recall:.3f})')

plt.title('Model Recall Across Sequential Data Cuts', fontsize=14)
plt.xlabel('Data Cut Number (Time/Sequence)', fontsize=12)
plt.ylabel('Recall Score', fontsize=12)
plt.ylim(0.35, 0.6) # Maintain consistent Y-limit
plt.legend()

# Adjust layout to prevent overlap and show the plots
plt.tight_layout()
plt.show()
#

In [ ]:
results_df['precision'].mean()

In [ ]:
model = joblib.load('dump_model.pk1')
old_model = model
old_params = model.get_params()
old_booster = old_model.get_booster()
features = old_booster.feature_names

In [ ]:
start_idx = 0
end_idx = 500
results_list = []

for cuts in range(cut_numbers):
    #define the split
    end_idx = start_idx + cut_len
    current_split = new_df_trades[start_idx:end_idx]

    if cuts < 1:
        #test model for each split
        base_precision, base_recall = test_model(df = current_split , model = model, features = features)
        
        results_list.append({
            'cut_number': cuts,
            'start_idx': start_idx,
            'end_Index': end_idx,
            'num_nows': len(current_split),
            'precision': base_precision,
            'recall': base_recall
        })

    
    if cuts > 0:
        train_split = new_df_trades[: (end_idx - 500)]
        X_train = train_split[features]
        y_train = train_split['Trades']

        old_model = model 
        old_params = old_model.get_params()
        old_booster = old_model.get_booster()
        model = XGBClassifier(**old_params)
        model.fit(X_train, y_train, xgb_model = old_booster)
        

        base_precision, base_recall = test_model(df = current_split , model = model, features = features)
        
        results_list.append({
            'cut_number': cuts,
            'start_idx': start_idx,
            'end_Index': end_idx,
            'num_nows': len(current_split),
            'precision': base_precision,
            'recall': base_recall
        })

        print(f" Cut {cuts+1}: Precision={base_precision:.4f}, Recall={base_recall:.4f}")






    start_idx = end_idx
    
    results_df = pd.DataFrame(results_list)

In [ ]:
results_df['precision'].mean()